In [29]:
# 1,2.) Create PySpark DataFrame using multiple lists
list_A = [1, 2, 3, 4, 5]
list_B = [4, 5, 6, 7, 8]

df = spark.createDataFrame(zip(list_A, list_B), ["A", "B"])
df.show()


+---+---+
|  A|  B|
+---+---+
|  1|  4|
|  2|  5|
|  3|  6|
|  4|  7|
|  5|  8|
+---+---+



In [30]:
# 3. How to get the items not common to both list A and list B?
list_A = [1, 2, 3, 4, 5]
list_B = [4, 5, 6, 7, 8]

common_items = list(set(list_A).symmetric_difference(set(list_B)))
print(common_items)


[1, 2, 3, 6, 7, 8]


In [31]:
# 4.) find the in a column find the frequency counts of unique items
from pyspark.sql.functions import count

data = [
    ("John", "Engineer"),
    ("John", "Engineer"),
    ("Mary", "Scientist"),
    ("Bob", "Engineer"),
    ("Bob", "Engineer"),
    ("Bob", "Scientist"),
    ("Sam", "Doctor")
]

df = spark.createDataFrame(data, ["name", "job"])
df.groupBy("job").agg(count("*").alias("count")).show()


+---------+-----+
|      job|count|
+---------+-----+
| Engineer|    4|
|Scientist|    2|
|   Doctor|    1|
+---------+-----+



In [32]:
# 5.) keep only top 2 most frequent values as it is and replace everything else as ‘Other’

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when

spark = SparkSession.builder.appName("TopJobs").getOrCreate()
data = [
    ("John", "Engineer"),
    ("John", "Engineer"),
    ("Mary", "Scientist"),
    ("Bob", "Engineer"),
    ("Bob", "Engineer"),
    ("Bob", "Scientist"),
    ("Sam", "Doctor")
]

df = spark.createDataFrame(data, ["name", "job"])
job_counts = df.groupBy("job").agg(count("*").alias("count"))
top_jobs = [row["job"] for row in job_counts.orderBy(col("count").desc()).limit(2).collect()]
df_final = df.withColumn(
    "job",
    when(col("job").isin(top_jobs), col("job")).otherwise("Other")
)

df_final.show()

+----+---------+
|name|      job|
+----+---------+
|John| Engineer|
|John| Engineer|
|Mary|Scientist|
| Bob| Engineer|
| Bob| Engineer|
| Bob|Scientist|
| Sam|    Other|
+----+---------+



In [33]:
# 6.) Rename columns using two lists

old_names = ["col1", "col2", "col3"]
new_names = ["new_col1", "new_col2", "new_col3"]

data = [(1, 2, 3), (4, 5, 6)]
df = spark.createDataFrame(data, old_names)
for old, new in zip(old_names, new_names):
    df = df.withColumnRenamed(old, new)

df.show()

+--------+--------+--------+
|new_col1|new_col2|new_col3|
+--------+--------+--------+
|       1|       2|       3|
|       4|       5|       6|
+--------+--------+--------+



In [34]:
# 7.) Find numbers that are multiples of 3
from pyspark.sql.functions import col, when
data = [
    (0, 7), (1, 6), (2, 9), (3, 7), (4, 3),
    (5, 8), (6, 9), (7, 8), (8, 3), (9, 8)
]

df = spark.createDataFrame(data, ["id", "random"])
df = df.withColumn("is_multiple_of_3", when(col("random") % 3 == 0, 1).otherwise(0))
df.show()

+---+------+----------------+
| id|random|is_multiple_of_3|
+---+------+----------------+
|  0|     7|               0|
|  1|     6|               1|
|  2|     9|               1|
|  3|     7|               0|
|  4|     3|               1|
|  5|     8|               0|
|  6|     9|               1|
|  7|     8|               0|
|  8|     3|               1|
|  9|     8|               0|
+---+------+----------------+



In [35]:
# 8.) Capitalize first letter of each name
from pyspark.sql.functions import col, initcap

data = [("john",), ("alice",), ("bob",)]
df = spark.createDataFrame(data, ["name"])
df = df.withColumn("name", initcap(col("name")))
df.show()

+-----+
| name|
+-----+
| John|
|Alice|
|  Bob|
+-----+



In [36]:
# 9.) Calculate number of characters in each word
from pyspark.sql.functions import col, length
data = [("john",), ("alice",), ("bob",)]
df = spark.createDataFrame(data, ["name"])
df = df.withColumn("word_length", length(col("name")))
df.show()

+-----+-----------+
| name|word_length|
+-----+-----------+
| john|          4|
|alice|          5|
|  bob|          3|
+-----+-----------+



In [ ]:
# 10.) Count number of nulls per column
from pyspark.sql.functions import isnan

data = [
    ("A", 1, None),
    ("B", None, 123),
    ("B", 3, 456),
    ("D", None, None)
]

df = spark.createDataFrame(data, ["Name", "Value", "id"])
null_counts = {col_name: df.filter(col(col_name).isNull()).count() for col_name in df.columns}
print(null_counts)